<br>

<img src="https://lindas.admin.ch/lindaslogo2.png" style="width:25%; float:right">

# LINDAS Tutorial

This page serves as a introductory tutorial to the LINDAS ecosystem. LINDAS is the Linked Data ecosystem of the Swiss Federal Administration.



# Introduction

The webpage you are currently viewing is a so called **interactive JupyterLite notebook**. In this notebook, you can change interactively the content of the single cells and execute these cells directly seeing the result of your changes immediately. The cells contain either [Markdown](https://en.wikipedia.org/wiki/Markdown) content (like this cell) or executable Python source code.

JupyterLite stems from JupyterLab with the advantage of being completely browser based without any backend infrastructure. This means that the execution of the cells could take some time during first execution. Subsequent executions will be much faster because of stored data in your browser cache.

If you are unfamiliar with the handling of Jupyter notebooks, here are two useful resources:

- [The JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [The Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

# Preliminaries

In this part some preliminaries for querying LINDAS with SPARQL are introduced. If you are only interested in the actual LINDAS tutorial, you can skip the whole section and start [here](#Actual-Tutorial).

## Install and Import the Necessary Modules

Querying a SPARQL endpoint is basically making a POST request to the corresponding endpoint URL. As JupyterLite at the moment has no support for Python's `requests` modul, the JavaScript fetch API is used (with some tricks). To make this happen, the following modules have to be importet: 

In [1]:
%pip install -q folium

In [2]:
import json
import pandas as pd
import numpy as np
import folium
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

## Define Main Query Function

As the JavaScript fetch API is asynchronous, the corresponding Python function `query` has to be declared as `async`. This function allows to query the 3 Swiss governmental triple stores.

In [3]:
async def query(query_string, store = "L"):
    
    # three Swiss triplestores
    if store == "F":
        address = 'https://fedlex.data.admin.ch/sparqlendpoint'
    elif store == "G":
        address = 'https://geo.ld.admin.ch/query'
    else:
        address = 'https://ld.admin.ch/query'
    
    # try the Post request with help of JS fetch
    # the creation of the request header is a little bit complicated because it needs to be a 
    # JavaScript JSON that is made within a Python source code
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "text/csv" })),
        )
    except:
        raise RuntimeError("fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        # ld.admin.ch throws errors starting with '{"message":'
        if '{"message":' in res:
            error = json.loads(res)
            raise RuntimeError("SPARQL query malformed: " + error["message"])
        # geo.ld.admin.ch throws errors starting with 'Parse error:'
        elif 'Parse error:' in res:
            raise RuntimeError("SPARQL query malformed: " + res)
        else:
            # if everything works out, create a pandas dataframe from the csv result
            df = pd.read_csv(StringIO(res))
            return df
    else:
        # fedlex.data.admin.ch throws error with response status 400
        if resp.status == 400:
            raise RuntimeError("Response status 400: Possible malformed SPARQL query. No syntactic advice available.")
        else:
            raise RuntimeError("Response status " + resp.status)

If you are interested in the details of using the JavaScript fetch API within JupyterLite, please consult:

- https://pyodide.org/en/stable/usage/faq.html#how-can-i-use-fetch-with-optional-arguments-from-python
- https://github.com/jupyterlite/jupyterlite/discussions/412
- https://lwebapp.com/en/post/pyodide-fetch

## Define Display Function

Displays pandas dataframe resulting from the SPARQL query as HTML with clickable links.

In [4]:
def display_result(df):
    df = HTML(df.to_html(render_links=True, escape=False))
    display(df)

# Questions

* Ist schema:parentOrganization von https://ld.admin.ch/ou/10002244 tatsächlich https://ld.admin.ch/ou/10008740? Scheint mir zweimal das gleiche zu sein...
* schema:endDate von https://ld.admin.ch/ou/20014808/2023-01-01 ist 2023-01-07, der vl:successor ist aber https://ld.admin.ch/ou/20014808/2023-01-16 mit schema:startDate 2023-01-16: Was ist dazwischen passiert?

# Actual Tutorial

## Alle Organisationen mit Namen

In [7]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
        schema:name ?name.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

,org,name
0,https://ld.admin.ch/ou/10000031,Eidgenössisches Finanzdepartement
1,https://ld.admin.ch/ou/10-11-02,Pensionskasse PUBLICA
2,https://ld.admin.ch/ou/10002061,"Bund, Kantone und FL"
3,https://ld.admin.ch/ou/10000000,Bundesrat
4,https://ld.admin.ch/ou/10000001,"Eidgenössisches Departement für Verteidigung, Bevölkerungsschutz und Sport"
5,https://ld.admin.ch/ou/10000025,Bundeskanzlei
6,https://ld.admin.ch/ou/10000026,Bundesgericht
7,https://ld.admin.ch/ou/10000028,Eidgenössisches Departement für auswärtige Angelegenheiten
8,https://ld.admin.ch/ou/10000029,Eidgenössisches Departement des Innern
9,https://ld.admin.ch/ou/10000030,Eidgenössisches Justiz- und Polizeidepartement


## Alle Organisationen, die mehrere Versionen haben

In [30]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT ?id ?name ?versions WHERE {
    
    ?id schema:name ?name.
    
    FILTER(lang(?name) = 'de')
    FILTER(?number > 1)
        
    {
    SELECT ?id (GROUP_CONCAT(?version; separator=" ← ") AS ?versions) (COUNT(?version) AS ?number) WHERE {

      ?version a vl:Version;
          vl:identity ?id.

    } 
    GROUP BY ?id
    }
}
ORDER BY DESC(?number)

""")

display_result(df)

,id,name,versions
0,https://ld.admin.ch/ou/20046263,Business Architektur,"<a href=""https://ld.admin.ch/ou/20046263/2023-01-01 ← https://ld.admin.ch/ou/20046263/2023-01-04 ← https://ld.admin.ch/ou/20046263/2023-01-06 ← https://ld.admin.ch/ou/20046263/2023-01-09 ← https://ld.admin.ch/ou/20046263/2023-01-12 ← https://ld.admin.ch/ou/20046263/2023-01-14 ← https://ld.admin.ch/ou/20046263/2023-01-18 ← https://ld.admin.ch/ou/20046263/2023-01-23 ← https://ld.admin.ch/ou/20046263/2023-01-25 ← https://ld.admin.ch/ou/20046263/2023-01-27 ← https://ld.admin.ch/ou/20046263/2023-01-31"" target=""_blank"">https://ld.admin.ch/ou/20046263/2023-01-01 ← https://ld.admin.ch/ou/20046263/2023-01-04 ← https://ld.admin.ch/ou/20046263/2023-01-06 ← https://ld.admin.ch/ou/20046263/2023-01-09 ← https://ld.admin.ch/ou/20046263/2023-01-12 ← https://ld.admin.ch/ou/20046263/2023-01-14 ← https://ld.admin.ch/ou/20046263/2023-01-18 ← https://ld.admin.ch/ou/20046263/2023-01-23 ← https://ld.admin.ch/ou/20046263/2023-01-25 ← https://ld.admin.ch/ou/20046263/2023-01-27 ← https://ld.admin.ch/ou/20046263/2023-01-31"
1,https://ld.admin.ch/ou/20051606,Sprachen i,https://ld.admin.ch/ou/20051606/2023-01-03 ← https://ld.admin.ch/ou/20051606/2023-01-05 ← https://ld.admin.ch/ou/20051606/2023-01-07 ← https://ld.admin.ch/ou/20051606/2023-01-11 ← https://ld.admin.ch/ou/20051606/2023-01-13 ← https://ld.admin.ch/ou/20051606/2023-01-16 ← https://ld.admin.ch/ou/20051606/2023-01-20 ← https://ld.admin.ch/ou/20051606/2023-01-24 ← https://ld.admin.ch/ou/20051606/2023-01-26 ← https://ld.admin.ch/ou/20051606/2023-01-28
2,https://ld.admin.ch/ou/20037711,Cyber Intelligence,https://ld.admin.ch/ou/20037711/2023-01-01 ← https://ld.admin.ch/ou/20037711/2023-01-03 ← https://ld.admin.ch/ou/20037711/2023-01-06 ← https://ld.admin.ch/ou/20037711/2023-01-11 ← https://ld.admin.ch/ou/20037711/2023-01-14 ← https://ld.admin.ch/ou/20037711/2023-01-16 ← https://ld.admin.ch/ou/20037711/2023-01-24 ← https://ld.admin.ch/ou/20037711/2023-01-27 ← https://ld.admin.ch/ou/20037711/2023-01-29
3,https://ld.admin.ch/ou/20037443,Cyber Defence,https://ld.admin.ch/ou/20037443/2023-01-01 ← https://ld.admin.ch/ou/20037443/2023-01-03 ← https://ld.admin.ch/ou/20037443/2023-01-06 ← https://ld.admin.ch/ou/20037443/2023-01-11 ← https://ld.admin.ch/ou/20037443/2023-01-14 ← https://ld.admin.ch/ou/20037443/2023-01-16 ← https://ld.admin.ch/ou/20037443/2023-01-22 ← https://ld.admin.ch/ou/20037443/2023-01-24 ← https://ld.admin.ch/ou/20037443/2023-01-27
4,https://ld.admin.ch/ou/20031558,Transformation,https://ld.admin.ch/ou/20031558/2023-01-01 ← https://ld.admin.ch/ou/20031558/2023-01-16 ← https://ld.admin.ch/ou/20031558/2023-01-20 ← https://ld.admin.ch/ou/20031558/2023-01-24 ← https://ld.admin.ch/ou/20031558/2023-01-26 ← https://ld.admin.ch/ou/20031558/2023-01-28
5,https://ld.admin.ch/ou/443568947,PB Schalter/Service Center,https://ld.admin.ch/ou/443568947/2023-01-01 ← https://ld.admin.ch/ou/443568947/2023-01-04 ← https://ld.admin.ch/ou/443568947/2023-01-06 ← https://ld.admin.ch/ou/443568947/2023-01-09 ← https://ld.admin.ch/ou/443568947/2023-01-20
6,https://ld.admin.ch/ou/20005626,NAZ und Ereignisbewältigung,https://ld.admin.ch/ou/20005626/2023-01-01 ← https://ld.admin.ch/ou/20005626/2023-01-02 ← https://ld.admin.ch/ou/20005626/2023-01-07 ← https://ld.admin.ch/ou/20005626/2023-01-13
7,https://ld.admin.ch/ou/30000073-10-03,"Business Solutions (MD, Vigilanzen)",https://ld.admin.ch/ou/30000073-10-03/2023-01-01 ← https://ld.admin.ch/ou/30000073-10-03/2023-01-04 ← https://ld.admin.ch/ou/30000073-10-03/2023-01-22
8,https://ld.admin.ch/ou/20050790,Grundlagen ZS und AUSB,https://ld.admin.ch/ou/20050790/2023-01-01 ← https://ld.admin.ch/ou/20050790/2023-01-02 ← https://ld.admin.ch/ou/20050790/2023-01-07
9,https://ld.admin.ch/ou/60032025-11-05,AVOR/PPS,https://ld.admin.ch/ou/60032025-11-05/2023-01-01 ← https://ld.admin.ch/ou/60032025-11-05/2023-01-03 ← https://ld.admin.ch/ou/60032025-11-05/2023-01-18
